# 🎙️ NLP Assignment 2: Low-Resource Speech-to-Text Translation

**Student Name**: (John) Paul Nagle  
**Student ID**: R00065426  

## Irish → English Cascaded Pipeline

**Module:** Natural Language Processing (MSc AI)  
**Deadline:** 10th May 2026  
**Weighting:** 50% of module

---

This notebook will guide you through building a cascaded **Automatic Speech Recognition (ASR) → Machine Translation (MT)** pipeline for translating Irish speech into English text.

### How to use this notebook
- Cells marked `# ✅ PROVIDED` are complete — run them as-is.
- Cells marked `# 📝 YOUR CODE HERE` require you to write your own code.
- Markdown cells with 💡 are hints. Read them before coding.
- Don't skip sections — each depends on the previous.

### Recommended runtime
Go to **Runtime → Change runtime type → T4 GPU** before starting.

---

### Table of Contents
1. [Setup & Installation](#setup)
2. [Dataset Loading & Exploration](#data)
3. [Audio Preprocessing](#audio)
4. [Baseline ASR — Whisper](#asr)
5. [Machine Translation — NLLB](#mt)
6. [Full Pipeline](#pipeline)
7. [Evaluation](#eval)
8. [Error Analysis](#error)
9. [Improved System](#improved)
10. [Final Results Table](#results)

---
## Section 1: Setup & Installation <a id='setup'></a>

In [ ]:
# ✅ PROVIDED — Install all required libraries
# This may take 2-3 minutes on first run.

!pip install -q transformers datasets evaluate sacrebleu jiwer librosa soundfile torchaudio torch jupyter ipywidgets
!pip install -q accelerate

print('✅ Libraries installed.')

In [ ]:
# ✅ PROVIDED — Imports

import os
import json
import time
import random
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch

from pathlib import Path
from IPython.display import Audio, display

from transformers import ( WhisperProcessor, WhisperForConditionalGeneration, pipeline )

import evaluate

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Check if running in Colab
try:
  import google.colab
  IN_COLAB = True
  print("Running in colab...")
  from google.colab import drive
  drive.mount('/content/drive')

except:
  print("Running locally")
  IN_COLAB = False


---
## Section 2: Dataset Loading & Exploration <a id='data'></a>

The dataset is the **IWSLT 2026 Irish–English Speech Translation** data.

Repository: https://github.com/shashwatup9k/iwslt2026_ga-eng

**Dataset structure (once cloned):**
```
└──iwslt2026_ga-eng
    └── iwslt2025_ga-eng
        └── dev
            ├── txt
            └── wav
```

The `txt` directory contains the file `test.eng`, whoch contains the reference English translations.

The `wav` folder contains the Irish spoken clips.

In [ ]:
#  Set the correct paths to the data splits based on what you saw above.
DATA_ROOT = Path('iwslt2026_ga-eng')
AUDIO_DIR = DATA_ROOT / 'iwslt2025_ga-eng' / 'dev' / 'wav'
REF_FILE = DATA_ROOT  / 'iwslt2025_ga-eng' / 'dev' / 'txt' / 'dev.eng'

print('Audio dir exists:', AUDIO_DIR.exists())
print('References exist:', REF_FILE.exists())

In [ ]:
# Load the transcripts and references into Python lists.
# Print the first 5 examples of each so you can verify they look right.

with open(REF_FILE) as f:
    english_refs = [line.strip() for line in f if line.strip()]

# Count wav files
audio_files = sorted(list(AUDIO_DIR.glob('*.wav')))
num_audio = len(audio_files)

# Print dataset statistics
print(f'Number of references:  {len(english_refs)}')
print(f'Number of wav files:   {num_audio}')
print(f'Match: {"✅" if len(english_refs) == num_audio else "❌"}')
print()

# Print the first 3 examples (transcript + reference pairs)
for i in range(3):
    print(f'--- Example {i+1} ---')
    print(f'English: {english_refs[i]}')
    print()

In [ ]:
# ✅ PROVIDED — Get list of audio files

audio_files = sorted(list(AUDIO_DIR.glob('*.wav')))
print(f'Found {len(audio_files)} audio files.')
if audio_files:
    print('First 5:', [f.name for f in audio_files[:5]])

---
## Section 3: Audio Inspection & Preprocessing <a id='audio'></a>

Before feeding audio to an ASR model, you need to understand its properties:
- **Sample rate** — Whisper expects 16,000 Hz (16 kHz)
- **Duration** — very short or very long clips may cause issues
- **Format** — most models expect mono-channel float arrays

💡 **Tip:** Librosa's `load()` function can resample audio automatically.

In [ ]:
# ✅ PROVIDED — Inspect one audio file

if audio_files:
    sample_file = audio_files[0]
    waveform, sr = librosa.load(sample_file, sr=None)  # sr=None preserves original rate
    
    print(f'File:        {sample_file.name}')
    print(f'Sample rate: {sr} Hz')
    print(f'Duration:    {len(waveform)/sr:.2f} seconds')
    print(f'Shape:       {waveform.shape}')
    print(f'Min/Max:     {waveform.min():.3f} / {waveform.max():.3f}')
    
    # Listen to it!
    display(Audio(waveform, rate=sr))

In [ ]:
# A function to load and preprocess an audio file.
# Requirements:
#   - Resample to 16000 Hz (Whisper's required rate)
#   - Return a float32 numpy array
#   - Handle mono audio only

TARGET_SR = 16000

def load_audio(filepath, target_sr=TARGET_SR):
    """
    Load an audio file and resample to target_sr.
    Returns: numpy array (float32), sample rate
    """
    # Load audio file and resample to target sample rate
    # sr=target_sr automatically resamples if needed
    # mono=True ensures single channel audio
    wav, sr = librosa.load(filepath, sr=target_sr, mono=True)
    
    # Ensure the array is float32 (librosa returns float32 by default)
    wav = wav.astype(np.float32)
    
    return wav, sr

# Test your function
if audio_files:
    wav, sr = load_audio(audio_files[0])
    assert sr == TARGET_SR, f'Sample rate should be {TARGET_SR}, got {sr}'
    assert wav.dtype == np.float32, 'Array should be float32'
    print(f'✅ load_audio works. Duration: {len(wav)/sr:.2f}s')

In [ ]:
# Compute and report basic statistics for the audio dataset.
# Report:
#   - Total number of audio files
#   - Average, min, max duration in seconds
#   - Total hours of audio

# Loop over audio_files, load each one, collect durations

sample_files = audio_files[:]
durations = []

for f in sample_files:
    # Load file and compute duration in seconds
    wav, sr = load_audio(f)
    duration = len(wav) / sr  # Duration = samples / sample_rate
    durations.append(duration)

# Convert to numpy array for easier statistics
durations = np.array(durations)

# Print statistics
print('=== Audio Statistics ===')
print(f'Files sampled: {len(sample_files)}')
print(f'Average duration: {durations.mean():.2f} s')
print(f'Min duration: {durations.min():.2f} s')
print(f'Max duration: {durations.max():.2f} s')
print(f'Total duration (sample): {durations.sum():.2f} s ({durations.sum()/3600:.2f} hours)')


---
## Section 4: Baseline ASR — Whisper <a id='asr'></a>

**Whisper** (Radford et al., 2022) is a multilingual ASR model from OpenAI, trained on 680,000 hours of multilingual audio. It supports Irish and requires 16 kHz mono audio input.

### Available Whisper model sizes:

| Model | Parameters | Recommended for |
|-------|-----------|----------------|
| `whisper-tiny` | 39M | Quick testing |
| `whisper-base` | 74M | Baseline (fast) |
| `whisper-small` | 244M | Better quality |
| `whisper-medium` | 769M | Good baseline |
| `whisper-large-v3` | 1.5B | Strongest (slow on CPU) |

💡 **Recommendation for baseline:** Start with `openai/whisper-small` for a reasonable speed/quality tradeoff on Colab.

In [ ]:
# Load a Whisper model and processor.
# 
# Requirements:
#   - Choose a Whisper model appropriate for your baseline
#   - Load the processor and model
#   - Move model to DEVICE
#   - Justify your choice in a comment

# Choose a model appropriate for Irish ASR
# Load Irish-specific Whisper model
# ASR_MODEL_ID = 'eamonmckenna/whisper-small-ga-ie-4000'
# ASR_MODEL_ID = 'eamonmckenna/whisper-medium-ga-ie-2025-01-21'
# ASR_MODEL_ID = 'eamonmckenna/whisper-tiny-ga-IE-v2'
if IN_COLAB:
    ASR_MODEL_ID = 'openai/whisper-large-v3' 
    ASR_MODEL_ID_SPECIALISED = 'eamonmckenna/whisper-large-v3-ga-ie-2025-01-24'
else:
    ASR_MODEL_ID = 'openai/whisper-small'
    ASR_MODEL_ID_SPECIALISED = 'eamonmckenna/whisper-large-v3-ga-ie-2025-01-24'

print(f'Loading ASR model: {ASR_MODEL_ID}...')

whisper_processor = WhisperProcessor.from_pretrained(ASR_MODEL_ID)
whisper_model = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_ID)
whisper_model = whisper_model.to(DEVICE).eval()

print(f'✅ Loaded {ASR_MODEL_ID}')
print(f'   Parameters: {sum(p.numel() for p in whisper_model.parameters()):,}')
print(f'   Device: {DEVICE}')

In [ ]:
# A function to transcribe a single audio file using Whisper.
#
# Steps:
#   1. Load and preprocess audio
#   2. Create input features with the Whisper processor
#   3. Generate transcription tokens (force Irish language: forced_decoder_ids)
#   4. Decode the output tokens to text
#   5. Return the transcription string

def transcribe_with_whisper(audio_path, processor, model):
    # Step 1: Load and preprocess audio 
    waveform, sr = load_audio(audio_path)
    
    # Step 2: Create input features with the Whisper processor
    inputs = processor(waveform, sampling_rate=sr, return_tensors='pt')
    input_features = inputs.input_features.to(DEVICE)
    
    # Step 3: Generate transcription tokens
    # Note: Whisper doesn't support Irish ('ga'), so we let it auto-detect
    # The model will transcribe phonetically or as the closest supported language
    with torch.no_grad():
        generated_tokens = model.generate(
            input_features
        )
    
    # Step 4: Decode the output tokens to text
    transcription = processor.batch_decode(
        generated_tokens, 
        skip_special_tokens=True  # This removes special tokens like <|cy|>
    )[0]
    
    # Step 5: Return the transcription string
    return transcription.strip()

# Test on one file
result = transcribe_with_whisper(audio_files[0], whisper_processor, whisper_model)
print(f'Test transcription: {result}')

In [ ]:
# Run ASR on a subset of the development/test set.
#
# - Process the first MAX_SAMPLES audio files (start with 50 for speed)
# - Store transcriptions in a list
# - Track and print average processing time per file
#
# 💡 TIP: Use tqdm for a progress bar:
#   from tqdm import tqdm

from tqdm import tqdm
import time

if IN_COLAB:
    MAX_SAMPLES = len(sample_files)
else:
    MAX_SAMPLES = 50

asr_hypotheses = []  # List of transcriptions
asr_times = []

# Loop over audio files with progress bar
for audio_path in tqdm(audio_files[:MAX_SAMPLES], desc='Running ASR'):
    try:
        # Time the transcription
        start_time = time.time()
        
        # Transcribe the audio file
        transcription = transcribe_with_whisper(
            audio_path, 
            whisper_processor, 
            whisper_model
        )
        
        # Record processing time
        elapsed_time = time.time() - start_time
        asr_times.append(elapsed_time)
        
        # Store the transcription
        asr_hypotheses.append(transcription)
        
    except Exception as e:
        # Handle errors gracefully
        print(f'\nError processing {audio_path}: {e}')
        asr_hypotheses.append('')  # Append empty string on failure
        asr_times.append(0)

# Print summary statistics
print(f'\n=== ASR Results ===')
print(f'Total ASR hypotheses: {len(asr_hypotheses)}')
print(f'Successful transcriptions: {sum(1 for h in asr_hypotheses if h)}')
print(f'Failed transcriptions: {sum(1 for h in asr_hypotheses if not h)}')
print(f'Average time per file: {np.mean(asr_times):.2f}s')
print(f'Total processing time: {sum(asr_times):.2f}s ({sum(asr_times)/60:.2f} min)')

# Show a few examples
print(f'\n=== Sample Transcriptions ===')
for i in range(min(10, len(asr_hypotheses))):
    print(f'{i+1}. {asr_hypotheses[i]}')

### Load Specialized ASR Model

Now load the Irish-specialized Whisper model for comparison.

In [ ]:
# Load specialized Irish Whisper model
print(f'Loading specialized ASR model: {ASR_MODEL_ID_SPECIALISED}...')

whisper_processor_specialised = WhisperProcessor.from_pretrained(ASR_MODEL_ID_SPECIALISED)
whisper_model_specialised = WhisperForConditionalGeneration.from_pretrained(ASR_MODEL_ID_SPECIALISED)
whisper_model_specialised = whisper_model_specialised.to(DEVICE).eval()

print(f'✅ Loaded {ASR_MODEL_ID_SPECIALISED}')
print(f'   Parameters: {sum(p.numel() for p in whisper_model_specialised.parameters()):,}')
print(f'   Device: {DEVICE}')

### Run ASR with Specialized Model

In [ ]:
# Run ASR with specialized model
asr_hypotheses_specialised = []
asr_times_specialised = []

for audio_path in tqdm(audio_files[:MAX_SAMPLES], desc='Running Specialized ASR'):
    try:
        start_time = time.time()
        
        transcription = transcribe_with_whisper(
            audio_path,
            whisper_processor_specialised,
            whisper_model_specialised
        )
        
        elapsed_time = time.time() - start_time
        asr_times_specialised.append(elapsed_time)
        asr_hypotheses_specialised.append(transcription)
        
    except Exception as e:
        print(f'\nError processing {audio_path}: {e}')
        asr_hypotheses_specialised.append('')
        asr_times_specialised.append(0)

print(f'\n=== Specialized ASR Results ===')
print(f'Total hypotheses: {len(asr_hypotheses_specialised)}')
print(f'Successful: {sum(1 for h in asr_hypotheses_specialised if h)}')
print(f'Failed: {sum(1 for h in asr_hypotheses_specialised if not h)}')
print(f'Average time: {np.mean(asr_times_specialised):.2f}s')
print(f'Total time: {sum(asr_times_specialised):.2f}s ({sum(asr_times_specialised)/60:.2f} min)')

print(f'\n=== Sample Transcriptions ===')
for i in range(min(10, len(asr_hypotheses_specialised))):
    print(f'{i+1}. {asr_hypotheses_specialised[i]}')

---
## Section 5: Machine Translation — NLLB / OPUS-MT <a id='mt'></a>

**NLLB-200** (No Language Left Behind, Meta AI, 2022) is a multilingual MT model covering 200 languages, including Irish (`gle_Latn`).

**OPUS-MT** models from Helsinki-NLP are bilingual translation models and can be stronger for specific language pairs such as Irish → English.

### MT models used in this notebook:

| Model | Type | Notes |
|-------|------|-------|
| `facebook/nllb-200-distilled-600M` | Multilingual | Baseline MT model |
| `Helsinki-NLP/opus-mt-ga-en` | Bilingual | Pair-specific alternative for Irish → English |

💡 **Language codes:** NLLB uses Irish = `gle_Latn`, English = `eng_Latn`; OPUS-MT does not need NLLB language tags.


In [ ]:
# Load MT models and tokenisers.

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_MODEL_ID = 'facebook/nllb-200-distilled-600M'
OPUS_MODEL_ID = 'Helsinki-NLP/opus-mt-ga-en'

SRC_LANG = 'gle_Latn'  # Irish (Gaeilge) in Latin script
TGT_LANG = 'eng_Latn'  # English in Latin script

print(f'Loading MT model: {NLLB_MODEL_ID}...')
mt_tokenizer = AutoTokenizer.from_pretrained(
    NLLB_MODEL_ID,
    src_lang=SRC_LANG
)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_ID)

if hasattr(mt_model.generation_config, 'task'):
    delattr(mt_model.generation_config, 'task')

mt_model = mt_model.to(DEVICE).eval()

print(f'✅ Loaded {NLLB_MODEL_ID}')
print(f'   Parameters: {sum(p.numel() for p in mt_model.parameters()):,}')
print(f'   Source language: {SRC_LANG}')
print(f'   Target language: {TGT_LANG}')
print(f'   Device: {DEVICE}')

print(f'\nLoading MT model: {OPUS_MODEL_ID}...')
opus_tokenizer = AutoTokenizer.from_pretrained(OPUS_MODEL_ID)
opus_model = AutoModelForSeq2SeqLM.from_pretrained(OPUS_MODEL_ID)
opus_model = opus_model.to(DEVICE).eval()

print(f'✅ Loaded {OPUS_MODEL_ID}')
print(f'   Parameters: {sum(p.numel() for p in opus_model.parameters()):,}')
print(f'   Device: {DEVICE}')


In [ ]:
# Translation helper functions.

def translate_with_nllb(text, tokenizer, model, src_lang=SRC_LANG, tgt_lang=TGT_LANG, max_new_tokens=256):
    """
    Translate Irish text to English with NLLB.
    Returns: translated string
    """
    if not text or not text.strip():
        return ''

    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = inputs.to(DEVICE)
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=max_new_tokens,
            num_beams=5,
            early_stopping=True
        )

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return translation.strip()


def translate_with_opus_mt(text, tokenizer, model, max_new_tokens=256):
    """
    Translate Irish text to English with OPUS-MT.
    Returns: translated string
    """
    if not text or not text.strip():
        return ''

    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    )
    inputs = inputs.to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=5,
            early_stopping=True
        )

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return translation.strip()

# Test both MT models
test_irish = 'Dia duit, conas atá tú?'
translation_nllb = translate_with_nllb(test_irish, mt_tokenizer, mt_model)
translation_opus = translate_with_opus_mt(test_irish, opus_tokenizer, opus_model)
print(f'Irish:        {test_irish}')
print(f'NLLB:         {translation_nllb}')
print(f'OPUS-MT:      {translation_opus}')


---
## Section 6: Full Cascaded Pipeline <a id='pipeline'></a>

Now combine ASR → MT into one end-to-end function.

```
Audio file
    │
    ▼ load_audio()
Waveform (16kHz)
    │
    ▼ transcribe_with_whisper()
Irish transcript (text)
    │
    ▼ translate_with_nllb() / translate_with_opus_mt()
English translation (text)
```


In [ ]:
# The full pipeline function.

def speech_to_text_translate(audio_path,
                              asr_processor, asr_model,
                              mt_tokenizer, mt_model,
                              translate_fn=translate_with_nllb):
    """
    Full cascaded speech translation: audio file → English text.
    """
    irish_transcript = transcribe_with_whisper(
        audio_path,
        asr_processor,
        asr_model
    )

    english_translation = translate_fn(
        irish_transcript,
        mt_tokenizer,
        mt_model
    )

    return irish_transcript, english_translation

# Test on a sample
if audio_files:
    transcript, translation = speech_to_text_translate(
        audio_files[0],
        whisper_processor,
        whisper_model,
        mt_tokenizer,
        mt_model,
        translate_fn=translate_with_nllb
    )
    print(f'ASR Transcript: {transcript}')
    print(f'MT Translation: {translation}')
    print(f'Reference:      {english_refs[0] if english_refs else "N/A"}')


In [ ]:
# Run the full pipeline on MAX_SAMPLES files with baseline ASR + NLLB MT.

from tqdm import tqdm
import time

if IN_COLAB:
    MAX_SAMPLES = len(audio_files)
else:
    MAX_SAMPLES = 50

baseline_transcripts = []
baseline_translations = []
failed_indices = []
processing_times = []

for idx, audio_path in enumerate(tqdm(audio_files[:MAX_SAMPLES], desc='Running Baseline Pipeline (ASR+NLLB)')):
    try:
        start_time = time.time()
        transcript, translation = speech_to_text_translate(
            audio_path,
            whisper_processor,
            whisper_model,
            mt_tokenizer,
            mt_model,
            translate_fn=translate_with_nllb
        )
        elapsed_time = time.time() - start_time
        processing_times.append(elapsed_time)
        baseline_transcripts.append(transcript)
        baseline_translations.append(translation)

        if not translation or not translation.strip():
            failed_indices.append(idx)

    except Exception as e:
        print(f'\n⚠️ Error processing file {idx} ({audio_path}): {e}')
        baseline_transcripts.append('')
        baseline_translations.append('')
        failed_indices.append(idx)
        processing_times.append(0)

print(f'\n=== Baseline Pipeline Results (ASR+NLLB) ===')
print(f'Processed: {len(baseline_translations)} files')
print(f'Successful: {len(baseline_translations) - len(failed_indices)} files')
print(f'Failed: {len(failed_indices)} files')
if processing_times:
    print(f'Average time per file: {np.mean(processing_times):.2f}s')
    print(f'Total processing time: {sum(processing_times):.2f}s ({sum(processing_times)/60:.2f} min)')

print(f'\n=== Sample Results ===')
for i in range(min(3, len(baseline_translations))):
    if i not in failed_indices:
        print(f'\n{i+1}. Transcript: {baseline_transcripts[i][:80]}...')
        print(f'   Translation: {baseline_translations[i][:80]}...')
        if english_refs and i < len(english_refs):
            print(f'   Reference: {english_refs[i][:80]}...')

import json

results = {
    'system_name': 'baseline_asr_nllb',
    'transcripts': baseline_transcripts,
    'translations': baseline_translations,
    'failed_indices': failed_indices,
    'processing_times': processing_times,
    'audio_files': [str(f) for f in audio_files[:MAX_SAMPLES]]
}

os.makedirs('results', exist_ok=True)
with open('results/baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print('✅ Results saved to results/baseline_results.json')


### Run Full Pipeline with Specialized ASR Model

In [ ]:
# Run full pipeline with specialized ASR + NLLB MT
specialised_transcripts = []
specialised_translations = []
specialised_failed_indices = []
specialised_processing_times = []

for idx, audio_path in enumerate(tqdm(audio_files[:MAX_SAMPLES], desc='Running Specialized Pipeline (ASR+NLLB)')):
    try:
        start_time = time.time()
        transcript, translation = speech_to_text_translate(
            audio_path,
            whisper_processor_specialised,
            whisper_model_specialised,
            mt_tokenizer,
            mt_model,
            translate_fn=translate_with_nllb
        )
        elapsed_time = time.time() - start_time
        specialised_processing_times.append(elapsed_time)
        specialised_transcripts.append(transcript)
        specialised_translations.append(translation)

        if not translation or not translation.strip():
            specialised_failed_indices.append(idx)

    except Exception as e:
        print(f'\n⚠️ Error processing file {idx} ({audio_path}): {e}')
        specialised_transcripts.append('')
        specialised_translations.append('')
        specialised_failed_indices.append(idx)
        specialised_processing_times.append(0)

print(f'\n=== Specialized Pipeline Results (ASR+NLLB) ===')
print(f'Processed: {len(specialised_translations)} files')
print(f'Successful: {len(specialised_translations) - len(specialised_failed_indices)} files')
print(f'Failed: {len(specialised_failed_indices)} files')
if specialised_processing_times:
    print(f'Average time per file: {np.mean(specialised_processing_times):.2f}s')
    print(f'Total processing time: {sum(specialised_processing_times):.2f}s ({sum(specialised_processing_times)/60:.2f} min)')

print(f'\n=== Sample Results ===')
for i in range(min(3, len(specialised_translations))):
    if i not in specialised_failed_indices:
        print(f'\n{i+1}. Transcript: {specialised_transcripts[i][:80]}...')
        print(f'   Translation: {specialised_translations[i][:80]}...')
        if english_refs and i < len(english_refs):
            print(f'   Reference: {english_refs[i][:80]}...')

specialised_results = {
    'system_name': 'specialised_asr_nllb',
    'transcripts': specialised_transcripts,
    'translations': specialised_translations,
    'failed_indices': specialised_failed_indices,
    'processing_times': specialised_processing_times,
    'audio_files': [str(f) for f in audio_files[:MAX_SAMPLES]]
}

with open('results/specialised_results.json', 'w', encoding='utf-8') as f:
    json.dump(specialised_results, f, ensure_ascii=False, indent=2)

print('✅ Results saved to results/specialised_results.json')


In [ ]:
# ✅ PROVIDED — Save baseline outputs

os.makedirs('results', exist_ok=True)

with open('results/baseline_hypotheses.txt', 'w') as f:
    for line in baseline_translations:
        f.write(line + '\n')

with open('results/references.txt', 'w') as f:
    for line in english_refs[:MAX_SAMPLES]:
        f.write(line + '\n')

print('✅ Saved results/baseline_hypotheses.txt')
print('✅ Saved results/references.txt')


### Run Full Pipeline with Baseline ASR + OPUS-MT


In [ ]:
# Run full pipeline with baseline ASR + OPUS-MT
baseline_opus_transcripts = []
baseline_opus_translations = []
baseline_opus_failed_indices = []
baseline_opus_processing_times = []

for idx, audio_path in enumerate(tqdm(audio_files[:MAX_SAMPLES], desc='Running Baseline Pipeline (ASR+OPUS)')):
    try:
        start_time = time.time()
        transcript, translation = speech_to_text_translate(
            audio_path,
            whisper_processor,
            whisper_model,
            opus_tokenizer,
            opus_model,
            translate_fn=translate_with_opus_mt
        )
        elapsed_time = time.time() - start_time
        baseline_opus_processing_times.append(elapsed_time)
        baseline_opus_transcripts.append(transcript)
        baseline_opus_translations.append(translation)

        if not translation or not translation.strip():
            baseline_opus_failed_indices.append(idx)

    except Exception as e:
        print(f'\n⚠️ Error processing file {idx} ({audio_path}): {e}')
        baseline_opus_transcripts.append('')
        baseline_opus_translations.append('')
        baseline_opus_failed_indices.append(idx)
        baseline_opus_processing_times.append(0)

print(f'\n=== Baseline Pipeline Results (ASR+OPUS-MT) ===')
print(f'Processed: {len(baseline_opus_translations)} files')
print(f'Successful: {len(baseline_opus_translations) - len(baseline_opus_failed_indices)} files')
print(f'Failed: {len(baseline_opus_failed_indices)} files')
if baseline_opus_processing_times:
    print(f'Average time per file: {np.mean(baseline_opus_processing_times):.2f}s')
    print(f'Total processing time: {sum(baseline_opus_processing_times):.2f}s ({sum(baseline_opus_processing_times)/60:.2f} min)')

baseline_opus_results = {
    'system_name': 'baseline_asr_opus',
    'transcripts': baseline_opus_transcripts,
    'translations': baseline_opus_translations,
    'failed_indices': baseline_opus_failed_indices,
    'processing_times': baseline_opus_processing_times,
    'audio_files': [str(f) for f in audio_files[:MAX_SAMPLES]]
}

with open('results/baseline_opus_results.json', 'w', encoding='utf-8') as f:
    json.dump(baseline_opus_results, f, ensure_ascii=False, indent=2)

print('✅ Results saved to results/baseline_opus_results.json')


### Run Full Pipeline with Specialized ASR + OPUS-MT


In [ ]:
# Run full pipeline with specialized ASR + OPUS-MT
specialised_opus_transcripts = []
specialised_opus_translations = []
specialised_opus_failed_indices = []
specialised_opus_processing_times = []

for idx, audio_path in enumerate(tqdm(audio_files[:MAX_SAMPLES], desc='Running Specialized Pipeline (ASR+OPUS)')):
    try:
        start_time = time.time()
        transcript, translation = speech_to_text_translate(
            audio_path,
            whisper_processor_specialised,
            whisper_model_specialised,
            opus_tokenizer,
            opus_model,
            translate_fn=translate_with_opus_mt
        )
        elapsed_time = time.time() - start_time
        specialised_opus_processing_times.append(elapsed_time)
        specialised_opus_transcripts.append(transcript)
        specialised_opus_translations.append(translation)

        if not translation or not translation.strip():
            specialised_opus_failed_indices.append(idx)

    except Exception as e:
        print(f'\n⚠️ Error processing file {idx} ({audio_path}): {e}')
        specialised_opus_transcripts.append('')
        specialised_opus_translations.append('')
        specialised_opus_failed_indices.append(idx)
        specialised_opus_processing_times.append(0)

print(f'\n=== Specialized Pipeline Results (ASR+OPUS-MT) ===')
print(f'Processed: {len(specialised_opus_translations)} files')
print(f'Successful: {len(specialised_opus_translations) - len(specialised_opus_failed_indices)} files')
print(f'Failed: {len(specialised_opus_failed_indices)} files')
if specialised_opus_processing_times:
    print(f'Average time per file: {np.mean(specialised_opus_processing_times):.2f}s')
    print(f'Total processing time: {sum(specialised_opus_processing_times):.2f}s ({sum(specialised_opus_processing_times)/60:.2f} min)')

specialised_opus_results = {
    'system_name': 'specialised_asr_opus',
    'transcripts': specialised_opus_transcripts,
    'translations': specialised_opus_translations,
    'failed_indices': specialised_opus_failed_indices,
    'processing_times': specialised_opus_processing_times,
    'audio_files': [str(f) for f in audio_files[:MAX_SAMPLES]]
}

with open('results/specialised_opus_results.json', 'w', encoding='utf-8') as f:
    json.dump(specialised_opus_results, f, ensure_ascii=False, indent=2)

print('✅ Results saved to results/specialised_opus_results.json')


---
## Section 7: Evaluation <a id='eval'></a>

We evaluate using two standard metrics:

| Metric | Description | Why use it? |
|--------|-------------|-------------|
| **BLEU** | N-gram overlap between hypothesis and reference | Standard MT metric |
| **chrF++** | Character + word n-gram F-score | More robust for morphologically rich languages |

💡 Always use **SacreBLEU** for reproducible BLEU scores — never implement BLEU from scratch.

In [ ]:
# ✅ PROVIDED — Evaluation helper functions

import sacrebleu

def compute_bleu(hypotheses, references):
    """
    Compute corpus BLEU using SacreBLEU.
    Args:
        hypotheses: list of predicted strings
        references: list of reference strings
    Returns: BLEU score (float)
    """
    # SacreBLEU expects references wrapped in a list of lists
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    return bleu.score

def compute_chrf(hypotheses, references):
    """
    Compute chrF++ using SacreBLEU.
    """
    chrf = sacrebleu.corpus_chrf(hypotheses, [references], word_order=2)
    return chrf.score

def compute_coverage(hypotheses):
    """Proportion of non-empty outputs."""
    n_empty = sum(1 for h in hypotheses if not h.strip())
    return 1.0 - (n_empty / max(len(hypotheses), 1))

def compute_repetition_rate(hypotheses, threshold=0.3):
    """Proportion of outputs where the same word appears too frequently."""
    flagged = 0
    for h in hypotheses:
        words = h.split()
        if len(words) < 4:
            continue
        from collections import Counter
        freqs = Counter(words)
        if freqs.most_common(1)[0][1] / len(words) > threshold:
            flagged += 1
    return flagged / max(len(hypotheses), 1)

print('✅ Evaluation functions loaded.')

In [ ]:
# Evaluate and compare all systems.

from evaluate import load
import pandas as pd

refs_subset = english_refs[:MAX_SAMPLES]
sacrebleu_metric = load('sacrebleu')
chrf_metric = load('chrf')


def has_repetition(text, n=3):
    """Check if text has repeated n-grams."""
    words = text.lower().split()
    if len(words) < n * 2:
        return False
    ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
    return len(ngrams) != len(set(ngrams))


def evaluate_system(system_name, translations, transcripts, processing_times):
    valid_hyps = []
    valid_refs = []

    for hyp, ref in zip(translations, refs_subset):
        if hyp and hyp.strip() and ref and ref.strip():
            valid_hyps.append(hyp.strip())
            valid_refs.append(ref.strip())

    bleu = 0.0
    chrf = 0.0
    rep_rate = 0.0
    avg_hyp_len = 0.0
    avg_ref_len = 0.0
    length_ratio = 0.0

    if valid_hyps:
        bleu = sacrebleu_metric.compute(
            predictions=valid_hyps,
            references=[[ref] for ref in valid_refs]
        )['score']

        chrf = chrf_metric.compute(
            predictions=valid_hyps,
            references=valid_refs,
            word_order=2
        )['score']

        repetitions = sum(1 for hyp in valid_hyps if has_repetition(hyp))
        rep_rate = repetitions / len(valid_hyps)
        avg_hyp_len = np.mean([len(h.split()) for h in valid_hyps])
        avg_ref_len = np.mean([len(r.split()) for r in valid_refs])
        length_ratio = avg_hyp_len / avg_ref_len if avg_ref_len > 0 else 0.0

    coverage = len(valid_hyps) / len(translations) if translations else 0.0
    avg_time = np.mean(processing_times) if processing_times else 0.0
    total_time = sum(processing_times) if processing_times else 0.0

    print(f'\n=== {system_name} ===')
    print(f'Total samples:        {len(translations)}')
    print(f'Valid samples:        {len(valid_hyps)}')
    print(f'Filtered out:         {len(translations) - len(valid_hyps)}')
    print(f'BLEU:                 {bleu:.2f}')
    print(f'chrF++:               {chrf:.2f}')
    print(f'Coverage:             {coverage:.1%}')
    print(f'Repetition rate:      {rep_rate:.1%}')
    print(f'Avg hypothesis len:   {avg_hyp_len:.1f} words')
    print(f'Avg reference len:    {avg_ref_len:.1f} words')
    print(f'Length ratio:         {length_ratio:.2f}')
    print(f'Average time/file:    {avg_time:.2f}s')
    print(f'Total processing:     {total_time:.2f}s')

    return {
        'System': system_name,
        'Samples': len(translations),
        'Valid': len(valid_hyps),
        'BLEU': round(bleu, 2),
        'chrF++': round(chrf, 2),
        'Coverage': round(coverage * 100, 2),
        'Repetition %': round(rep_rate * 100, 2),
        'Avg Hyp Len': round(avg_hyp_len, 2),
        'Avg Ref Len': round(avg_ref_len, 2),
        'Length Ratio': round(length_ratio, 2),
        'Avg Time/File (s)': round(avg_time, 2),
        'Total Time (s)': round(total_time, 2),
    }

systems = [
    ('Baseline ASR + NLLB', baseline_translations, baseline_transcripts, processing_times),
    ('Specialized ASR + NLLB', specialised_translations, specialised_transcripts, specialised_processing_times),
    ('Baseline ASR + OPUS-MT', baseline_opus_translations, baseline_opus_transcripts, baseline_opus_processing_times),
    ('Specialized ASR + OPUS-MT', specialised_opus_translations, specialised_opus_transcripts, specialised_opus_processing_times),
]

evaluation_rows = [evaluate_system(*system) for system in systems]
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df = evaluation_df.sort_values(by=['BLEU', 'chrF++'], ascending=False).reset_index(drop=True)

print('\n=== Overall Comparison Table ===')
display(evaluation_df)


### Evaluate Specialized System

In [ ]:
# Compare all systems against the baseline NLLB system.

baseline_row = evaluation_df[evaluation_df['System'] == 'Baseline ASR + NLLB'].iloc[0]
comparison_df = evaluation_df.copy()
comparison_df['BLEU Δ vs Baseline'] = (comparison_df['BLEU'] - baseline_row['BLEU']).round(2)
comparison_df['chrF++ Δ vs Baseline'] = (comparison_df['chrF++'] - baseline_row['chrF++']).round(2)
comparison_df['Coverage Δ vs Baseline'] = (comparison_df['Coverage'] - baseline_row['Coverage']).round(2)
comparison_df['Time/File Δ vs Baseline'] = (comparison_df['Avg Time/File (s)'] - baseline_row['Avg Time/File (s)']).round(2)

print('=== Relative Comparison vs Baseline ASR + NLLB ===')
display(comparison_df[['System', 'BLEU', 'BLEU Δ vs Baseline', 'chrF++', 'chrF++ Δ vs Baseline', 'Coverage', 'Coverage Δ vs Baseline', 'Avg Time/File (s)', 'Time/File Δ vs Baseline']])


---
## Section 8: Error Analysis <a id='error'></a>

Quantitative metrics tell you *how much* a system gets wrong — error analysis tells you *why*.

### Common error types to look for:

| Error Type | Description | Example |
|-----------|-------------|--------|
| **ASR substitution** | Wrong word transcribed | "sráid" → "sraith" |
| **ASR deletion** | Missing words | "tá sé ann" → "sé ann" |
| **ASR hallucination** | Model confabulates plausible-sounding but wrong text | |
| **MT untranslated** | Irish word passed through to English output | |
| **MT mistranslation** | Wrong meaning | "leabhar" → "book" ✅ vs "leabhar" → "library" ❌ |
| **MT over-generation** | Unexplained extra text in output | |
| **Error propagation** | ASR error causes MT error | |

💡 Aim for a systematic analysis of at least **20 examples**.

In [ ]:
# ✅ PROVIDED — Side-by-side comparison helper

def show_examples(n=10, start=0):
    """Display examples in a readable format for error analysis."""
    for i in range(start, min(start + n, len(baseline_translations))):
        print(f'--- Example {i+1} ---')
        print(f'Irish transcript (ASR): {baseline_transcripts[i]}')
        print(f'English translation:    {baseline_translations[i]}')
        print(f'English reference:      {english_refs[i]}')
        print()

show_examples(n=5)

In [ ]:
# 📝 YOUR CODE HERE
# Task: Manually annotate 20+ examples with error categories.

# To complete this properly, you need to:

# Run the display code to see actual examples
# Manually review each example comparing:
# Audio → ASR transcript (check for ASR errors)
# ASR transcript → MT output (check for MT errors)
# MT output → Reference (check final quality)
# Update the annotations with real error types based on your observations
# Add more examples until you have 20-25 annotations
# Error type guidelines:

# ASR Errors:

# substitution: Wrong word transcribed
# deletion: Word missing from transcript
# insertion: Extra word added
# hallucination: Content not in audio
# language_confusion: Wrong language detected
# none: No errors
# MT Errors:

# mistranslation: Wrong meaning
# omission: Missing content
# addition: Extra content
# word_order: Awkward ordering
# literal_translation: Too literal, unnatural
# none: No errors

from collections import Counter
import random

# First, let's display some examples to analyze
print('=== Examples for Error Analysis ===\n')
num_examples = min(25, len(valid_hyps))

# Select examples (mix of random and first few)
example_indices = list(range(min(15, len(valid_hyps)))) + \
                  random.sample(range(15, len(valid_hyps)), min(10, len(valid_hyps)-15))

for i in example_indices[:5]:  # Show first 5 for reference
    print(f'Example {i}:')
    if i < len(baseline_transcripts):
        print(f'  ASR Output: {baseline_transcripts[i][:100]}...')
    print(f'  MT Output:  {valid_hyps[i][:100]}...')
    print(f'  Reference:  {valid_refs[i][:100]}...')
    print()

# Error type definitions:
# ASR errors: substitution, deletion, insertion, hallucination, language_confusion
# MT errors: mistranslation, omission, addition, word_order, literal_translation, none

# Manual annotations (you need to fill these based on actual outputs)
error_annotations = [
    {
        'id': 0,
        'asr_errors': ['language_confusion'],  # Whisper detected wrong language
        'mt_errors': ['mistranslation'],
        'notes': 'ASR confused Irish with Welsh/English. MT struggled with mixed input.'
    },
    {
        'id': 1,
        'asr_errors': ['substitution'],
        'mt_errors': ['none'],
        'notes': 'Minor ASR substitution but MT produced acceptable translation.'
    },
    {
        'id': 2,
        'asr_errors': ['none'],
        'mt_errors': ['word_order'],
        'notes': 'ASR correct but MT reordered words awkwardly.'
    },
    {
        'id': 3,
        'asr_errors': ['deletion'],
        'mt_errors': ['omission'],
        'notes': 'ASR missed a word, cascaded into MT omission.'
    },
    {
        'id': 4,
        'asr_errors': ['hallucination'],
        'mt_errors': ['addition'],
        'notes': 'ASR added non-existent words, MT translated them.'
    },
    {
        'id': 5,
        'asr_errors': ['language_confusion'],
        'mt_errors': ['literal_translation'],
        'notes': 'ASR mixed languages, MT too literal.'
    },
    {
        'id': 6,
        'asr_errors': ['substitution', 'deletion'],
        'mt_errors': ['mistranslation'],
        'notes': 'Multiple ASR errors led to poor MT output.'
    },
    {
        'id': 7,
        'asr_errors': ['none'],
        'mt_errors': ['none'],
        'notes': 'Both ASR and MT performed well.'
    },
    {
        'id': 8,
        'asr_errors': ['insertion'],
        'mt_errors': ['addition'],
        'notes': 'ASR inserted extra words, MT translated them.'
    },
    {
        'id': 9,
        'asr_errors': ['language_confusion'],
        'mt_errors': ['mistranslation'],
        'notes': 'Whisper detected English instead of Irish.'
    },
    {
        'id': 10,
        'asr_errors': ['substitution'],
        'mt_errors': ['word_order'],
        'notes': 'ASR substitution + MT word order issue.'
    },
    {
        'id': 11,
        'asr_errors': ['none'],
        'mt_errors': ['literal_translation'],
        'notes': 'ASR good but MT too literal, unnatural English.'
    },
    {
        'id': 12,
        'asr_errors': ['deletion', 'substitution'],
        'mt_errors': ['omission', 'mistranslation'],
        'notes': 'Cascading errors from ASR to MT.'
    },
    {
        'id': 13,
        'asr_errors': ['hallucination'],
        'mt_errors': ['addition'],
        'notes': 'ASR hallucinated content not in audio.'
    },
    {
        'id': 14,
        'asr_errors': ['language_confusion'],
        'mt_errors': ['none'],
        'notes': 'ASR confused language but MT handled it reasonably.'
    },
    {
        'id': 15,
        'asr_errors': ['substitution'],
        'mt_errors': ['mistranslation'],
        'notes': 'ASR error propagated to MT.'
    },
    {
        'id': 16,
        'asr_errors': ['none'],
        'mt_errors': ['word_order'],
        'notes': 'Good ASR, awkward MT word order.'
    },
    {
        'id': 17,
        'asr_errors': ['deletion'],
        'mt_errors': ['omission'],
        'notes': 'ASR deletion caused MT omission.'
    },
    {
        'id': 18,
        'asr_errors': ['language_confusion', 'substitution'],
        'mt_errors': ['mistranslation'],
        'notes': 'Multiple ASR issues, poor MT result.'
    },
    {
        'id': 19,
        'asr_errors': ['insertion'],
        'mt_errors': ['addition'],
        'notes': 'ASR added words, MT translated them.'
    },
    {
        'id': 20,
        'asr_errors': ['none'],
        'mt_errors': ['literal_translation'],
        'notes': 'ASR correct, MT too literal.'
    },
]

# Compute error statistics
all_asr_errors = [e for ann in error_annotations for e in ann['asr_errors']]
all_mt_errors = [e for ann in error_annotations for e in ann['mt_errors']]

asr_counter = Counter(all_asr_errors)
mt_counter = Counter(all_mt_errors)

print('\n=== Error Analysis Summary ===')
print(f'Total examples analyzed: {len(error_annotations)}')
print(f'\nASR Error Types:')
for error_type, count in asr_counter.most_common():
    print(f'  {error_type:20s}: {count:2d} ({count/len(error_annotations)*100:.1f}%)')

print(f'\nMT Error Types:')
for error_type, count in mt_counter.most_common():
    print(f'  {error_type:20s}: {count:2d} ({count/len(error_annotations)*100:.1f}%)')

# Identify most common error patterns
print(f'\n=== Key Findings ===')
print(f'Most common ASR error: {asr_counter.most_common(1)[0][0] if asr_counter else "none"}')
print(f'Most common MT error: {mt_counter.most_common(1)[0][0] if mt_counter else "none"}')

# Count cascading errors (where ASR error leads to MT error)
cascading = sum(1 for ann in error_annotations 
                if 'none' not in ann['asr_errors'] and 'none' not in ann['mt_errors'])
print(f'Cascading errors (ASR→MT): {cascading} ({cascading/len(error_annotations)*100:.1f}%)')

# Perfect examples
perfect = sum(1 for ann in error_annotations 
              if ann['asr_errors'] == ['none'] and ann['mt_errors'] == ['none'])
print(f'Perfect translations: {perfect} ({perfect/len(error_annotations)*100:.1f}%)')


---
## Section 9: Improved System <a id='improved'></a>

You must implement **at least one meaningful improvement** over your baseline.

### Possible directions (choose one or more):

| Direction | Description | Effort |
|-----------|-------------|--------|
| 🔄 **Different ASR model** | Try `whisper-medium` or a Wav2Vec2 Irish model | Low |
| 🔄 **Different MT model** | Try `nllb-1.3B` or `Helsinki-NLP/opus-mt-ga-en` | Low |
| 🔧 **ASR language forcing** | Ensure Whisper transcribes in Irish, not English | Low |
| 🔧 **Beam search tuning** | Increase `num_beams` for MT | Low |
| ✂️ **Output filtering** | Remove empty/degenerate outputs before MT | Medium |
| 🧹 **Text normalisation** | Clean ASR output before MT (punctuation, case) | Medium |
| 🎛️ **Audio augmentation** | Normalise audio volume before ASR | Medium |

You must **justify your choice** in your report and **compare results** against the baseline.

In [ ]:
# Improved System: Using Irish-Specialized Whisper Model
# 
# Improvement: Replace baseline Whisper with eamonmckenna/whisper-large-v3-ga-ie-2025-01-24
# This model is fine-tuned specifically for Irish (Gaeilge) speech recognition,
# which should provide better transcription quality than the general-purpose Whisper model.

# The specialized model and pipeline have already been run earlier in the notebook.
# Here we assign the results to the improved_translations variable for evaluation.

improved_translations = specialised_translations

# Evaluate the improved system with the same metrics
improved_bleu = bleu_specialised
improved_chrf = chrf_specialised
improved_coverage = coverage_specialised
improved_rep_rate = rep_rate_specialised

print('\n=== Improved System (Specialized ASR) Results ===')
print(f'Model: {ASR_MODEL_ID_SPECIALISED}')
print(f'Samples evaluated:   {len(valid_hyps_specialised)}')
print(f'BLEU:                {improved_bleu:.2f}')
print(f'chrF++:              {improved_chrf:.2f}')
print(f'Coverage:            {improved_coverage:.1%}')
print(f'Repetition rate:     {improved_rep_rate:.1%}')

print(f'\n=== Improvement over Baseline ===')
print(f'BLEU improvement:   {improved_bleu - bleu:+.2f} points ({(improved_bleu/bleu - 1)*100:+.1f}%)')
print(f'chrF++ improvement: {improved_chrf - chrf:+.2f} points ({(improved_chrf/chrf - 1)*100:+.1f}%)')

---
## Section 10: Final Results Table <a id='results'></a>

Your report must include a results table. Build it here.

In [ ]:
# 📝 YOUR CODE HERE
# Task: Build and display a results table comparing baseline and improved system.

results = pd.DataFrame([
    {
        'System': f'Baseline ({ASR_MODEL_ID} + {MT_MODEL_ID})',
        'BLEU': bleu,
        'chrF++': chrf,
        'Coverage': f'{coverage:.1%}',
        'Rep. Rate': f'{rep_rate:.1%}',
    },
    {
        'System': f'Improved ({ASR_MODEL_ID_SPECIALISED} + {MT_MODEL_ID})',
        'BLEU': bleu_specialised,
        'chrF++': chrf_specialised,
        'Coverage': f'{coverage_specialised:.1%}',
        'Rep. Rate': f'{rep_rate_specialised:.1%}',
    }
])

results = results.set_index('System')
print('=== Final Results ===')
display(results)

# Save to file
results.to_csv('results/final_results.csv')
print('\nSaved to results/final_results.csv')

In [ ]:
# 📝 YOUR CODE HERE
# Task: Write a brief (3-5 sentence) interpretation of your results in this cell.
# This will help you draft the Results section of your report.

# TODO: Fill in your analysis
print("""
=== Results Interpretation ===

[Write 3-5 sentences here interpreting your results. For example:]
- How do BLEU/chrF++ compare between systems?
- Which errors are most common and why?
- Was the improvement meaningful? What might explain it?
- What are the key limitations of your approach?
""")

---
## ✅ Submission Checklist

Before submitting, confirm:

- [ ] All cells run top-to-bottom without errors
- [ ] `results/baseline_hypotheses.txt` saved
- [ ] `results/final_results.csv` saved
- [ ] Error analysis covers at least 20 examples
- [ ] Improved system is distinct from baseline and justified
- [ ] README.md explains how to run your notebook
- [ ] ACL-format report is complete (6–8 pages)
- [ ] All AI tool usage is declared in the report appendix
- [ ] All random seeds are set and documented

**Good luck! 🍀**